# Task A — Run 9 ensemble on train + labelled validation

This final-fit notebook combines the original Task A training rows with the released labelled validation rows, trains the second-best Run 9 recipe, and predicts the released Task A test rows. Both components are demojized: MuRIL uses its default demojization and the TF-IDF/SVM command passes `--demojize`.

Recipe: two MuRIL encoder layers reinitialized, six epochs, one seed, and a fixed 57% TF-IDF/SVM + 43% MuRIL blend at threshold 0.50. The output is for the released test file, not the closed CodaBench validation file.

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile, json
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only", "origin", "task-b"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1", "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

# Protect against a stale Kaggle clone that predates the released evaluation files.
from urllib.request import urlretrieve
required = ["data/raw/hastika_binary_validation.csv", "data/raw/hastika_binary_test.csv"]
raw_base = "https://raw.githubusercontent.com/robinpnalex/Hastika-ICON2026/task-b/"
for rel in required:
    path = pathlib.Path(rel)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print("downloading missing released data:", rel)
        urlretrieve(raw_base + rel, path)
    assert path.exists() and path.stat().st_size > 0, f"missing required data: {rel}"
print("released data files ready")

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "enable a GPU"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Combine labelled data and stage the released test file

In [ ]:
from pathlib import Path
train_path = Path("data/raw/binary_train.csv")
val_path = Path("data/raw/hastika_binary_validation.csv")
test_path = Path("data/raw/hastika_binary_test.csv")
combined_path = Path("data/derived/task_a_run9_train_plus_validation.csv")
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined = pd.concat([pd.read_csv(train_path), pd.read_csv(val_path)], ignore_index=True).drop_duplicates("id").reset_index(drop=True)
assert len(combined) == 7252 and combined["id"].is_unique
combined.to_csv(combined_path, index=False)
assert len(pd.read_csv(test_path)) == 806
print("combined labelled rows:", len(combined), "released test rows:", len(pd.read_csv(test_path)))
CANON_TRAIN = Path("data/raw/binary_train.csv")
CANON_TEST = Path("data/raw/binary_validation_inputs.csv")
shutil.copy2(CANON_TRAIN, CANON_TRAIN.with_suffix(".csv.before_run9"))
shutil.copy2(CANON_TEST, CANON_TEST.with_suffix(".csv.before_run9"))
shutil.copy2(combined_path, CANON_TRAIN)
shutil.copy2(test_path, CANON_TEST)


## 2. Train the two demojized Run 9 components

In [ ]:
SVM_TAG, MURIL_TAG = "a_run9_trainval_svm", "a_run9_trainval_muril"
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm", "--task", "a", "--tag", SVM_TAG, "--demojize", "--full-fit"], log=f"artifacts/logs/{SVM_TAG}.log")
run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", MURIL_TAG, "--model", "google/muril-base-cased", "--folds", "1", "--seeds", "42", "--epochs", "6", "--bs", "8", "--grad-accum", "2", "--eval-bs", "32", "--select", "last", "--reinit-layers", "2"], log=f"artifacts/logs/{MURIL_TAG}.log")
svm_p = np.load(Path("artifacts/runs") / SVM_TAG / "test_probs.npy")
muril_p = np.load(Path("artifacts/runs") / MURIL_TAG / "test_probs.npy")
assert svm_p.shape == muril_p.shape == (806, 2)
print("both components completed")


## 3. Blend, restore paths, and package

In [ ]:
W_SVM, THRESH = 0.57, 0.50
test = pd.read_csv(test_path)
p_hate = W_SVM * svm_p[:, 1] + (1 - W_SVM) * muril_p[:, 1]
pred_path = Path("artifacts/runs/task_a_run9_trainval_test/predictions.csv")
pred_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": test["id"], "label": np.where(p_hate > THRESH, "Hate", "Non-Hate")}).to_csv(pred_path, index=False)
shutil.move(str(CANON_TRAIN.with_suffix(".csv.before_run9")), str(CANON_TRAIN))
shutil.move(str(CANON_TEST.with_suffix(".csv.before_run9")), str(CANON_TEST))
ZIP = Path("/kaggle/working/task_a_run9_trainval_test.zip")
run([sys.executable, "-m", "hastika.common.submission", "--task", "a", "--pred", str(pred_path), "--inputs", str(test_path), "--out", str(ZIP)])
with zipfile.ZipFile(ZIP) as z: assert z.namelist() == ["predictions.csv"]
OUT = Path("/kaggle/working/task_a_run9_trainval_test_outputs"); OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / ZIP.name); shutil.copy2(pred_path, OUT / pred_path.name)
json.dump({"train_rows": len(combined), "svm_weight": W_SVM, "muril_weight": 1-W_SVM, "threshold": THRESH, "reinit_layers": 2, "demojized": True}, open(OUT / "config.json", "w"), indent=2)
print("READY TO DOWNLOAD:", ZIP)
